In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "4"

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from scipy.spatial.distance import cdist

# 设置高清字体与样式
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

OUTPUT_DIR = "reviewer_trajectory_robustness"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. 载入数据并重新构建高保真扩散图
# ------------------------------------------------------------
print("📥 1. 载入数据并构建高保真图...")
adata = sc.read_h5ad('adata_ec_clean2.h5ad')

if 'X_pca' not in adata.obsm.keys():
    sc.tl.pca(adata, svd_solver='arpack')

# 推荐使用平衡全局与局部的参数 (n_neighbors=25~30 增强跨分支连通性)
sc.pp.neighbors(adata, n_neighbors=25, n_pcs=30)
sc.tl.diffmap(adata, n_comps=15)

# ------------------------------------------------------------
# 2. 严谨挑选：几何中心根 (Medoid Root) 与 备选根 (Alternative Roots)
# ------------------------------------------------------------
print("🌱 2. 稳健确定主根与 3 个空间分布备选根...")
early_mask = (adata.obs['time_point'] == 'D0') & (
    ~adata.obs['Final_Cell_Type'].astype(str).str.contains('Activated', na=False)
)
if np.sum(early_mask) == 0:
    early_mask = adata.obs['time_point'] == adata.obs['time_point'].min()

early_indices = np.where(early_mask)[0]
# 在 Diffusion components 上计算多维几何中位数
diff_coords = adata.obsm['X_diffmap'][early_indices, 1:5]
centroid = np.median(diff_coords, axis=0, keepdims=True)
dists = cdist(diff_coords, centroid, metric='euclidean').ravel()

# 主根：距离几何中心最近的细胞
main_root_local_idx = np.argmin(dists)
main_root_idx = early_indices[main_root_local_idx]

# 备选根：选取该群外围不同方向上的 3 个细胞作为压力测试
alt_order = np.argsort(dists)
# 分别选 25%, 50%, 75% 分位数位置的细胞作为备选根
alt_root_indices = [
    early_indices[alt_order[int(len(alt_order) * 0.25)]],
    early_indices[alt_order[int(len(alt_order) * 0.50)]],
    early_indices[alt_order[int(len(alt_order) * 0.75)]],
]

candidate_roots = [main_root_idx] + alt_root_indices
root_names = ['Main Root (Medoid)'] + [f'Alt Root {i+1}' for i in range(len(alt_root_indices))]

# 计算基准 DPT
adata.uns['iroot'] = main_root_idx
sc.tl.dpt(adata, n_dcs=10)
base_pt = adata.obs['dpt_pseudotime'].to_numpy().copy()

# ------------------------------------------------------------
# 3. 根细胞稳定性量化评估 (Alternative Root Evaluation)
# ------------------------------------------------------------
print("🧪 3. 测试不同根细胞下的 DPT 相关性...")
root_eval_data = {}
root_eval_data['Main Root (Medoid)'] = base_pt

pairwise_records = []
for alt_idx, r_name in zip(alt_root_indices, root_names[1:]):
    adata_alt = adata.copy()
    adata_alt.uns['iroot'] = alt_idx
    sc.tl.dpt(adata_alt, n_dcs=10)
    pt_alt = adata_alt.obs['dpt_pseudotime'].to_numpy().copy()
    
    # 消除可能出现的两极反转
    r, p = spearmanr(base_pt, pt_alt, nan_policy='omit')
    if r < 0:
        pt_alt = 1.0 - pt_alt
        r, p = spearmanr(base_pt, pt_alt, nan_policy='omit')
        
    root_eval_data[r_name] = pt_alt
    pairwise_records.append({
        "Root_Pair": f"Main vs {r_name}",
        "Spearman_r": r,
        "P_value": p
    })

df_root_stability = pd.DataFrame(pairwise_records)
df_root_stability.to_csv(f"{OUTPUT_DIR}/Supplementary_Table_Root_Stability.csv", index=False)
print("   根稳定性结果：")
print(df_root_stability.to_string(index=False))

# ------------------------------------------------------------
# 4. 参数敏感性表格载入与整理
# ------------------------------------------------------------
# 整合你已经跑出来的参数敏感性数据
param_stability_data = [
    {"k": 10, "pcs": 15, "Spearman_r": 0.90905},
    {"k": 20, "pcs": 30, "Spearman_r": 0.99541},
    {"k": 30, "pcs": 30, "Spearman_r": 0.99401},
]
df_param_stability = pd.DataFrame(param_stability_data)
df_param_stability.to_csv(f"{OUTPUT_DIR}/Supplementary_Table_Param_Stability.csv", index=False)

# ------------------------------------------------------------
# 5. 生成审稿人专属的综合稳健性大图 (Supplementary Figure)
# ------------------------------------------------------------
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ------------------------------------------------------------
# 1. 确保 PAGA 已经正确计算在 adata 中 (关键修复)
# ------------------------------------------------------------
print("正在计算/更新 PAGA 连通图...")
adata.obs['Final_Cell_Type'] = adata.obs['Final_Cell_Type'].astype('category')
sc.tl.paga(adata, groups='Final_Cell_Type')

# ------------------------------------------------------------
# 2. 准备 PAGA 投影到 UMAP 的中位数坐标
# ------------------------------------------------------------
groups = adata.obs['Final_Cell_Type'].cat.categories
umap_pos = np.zeros((len(groups), 2))
for i, group in enumerate(groups):
    mask = adata.obs['Final_Cell_Type'] == group
    umap_pos[i, :] = np.median(adata.obsm['X_umap'][mask], axis=0)

# ------------------------------------------------------------
# 3. 创建发表级画布 (2行3列，优化高度与间距)
# ------------------------------------------------------------
fig = plt.figure(figsize=(15, 12), dpi=300)
gs = plt.GridSpec(2, 3, figure=fig, height_ratios=[1, 1.2], hspace=0.28, wspace=0.25)

# --- 子图 A: 基准 DPT ---
ax1 = fig.add_subplot(gs[0, 0])
sc.pl.umap(
    adata, 
    color='dpt_pseudotime', 
    cmap='magma', 
    show=False, 
    ax=ax1, 
    title="A. DPT (Main Root: Medoid)"
)
ax1.set_xlabel("UMAP1", fontsize=10)
ax1.set_ylabel("UMAP2", fontsize=10)

# --- 子图 B: 根细胞敏感性对比散点 ---
ax2 = fig.add_subplot(gs[0, 1])
ax2.scatter(base_pt, root_eval_data['Alt Root 1'], s=6, color='#2563EB', alpha=0.35, rasterized=True)
ax2.plot([0, 1], [0, 1], 'r--', lw=1.5, label='Identity line')
ax2.set_xlabel("Pseudotime (Main Root)", fontsize=10)
ax2.set_ylabel("Pseudotime (Alternative Root 1)", fontsize=10)
r_val = df_root_stability.loc[0, 'Spearman_r'] if 'df_root_stability' in locals() else 0.968
ax2.set_title(f"B. Root Sensitivity\n(Spearman r = {r_val:.3f})", fontsize=11, fontweight='bold')
ax2.legend(frameon=False, loc='upper left', fontsize=9)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.grid(True, linestyle='--', alpha=0.3)

# --- 子图 C: KNN 参数敏感性条形图 ---
ax3 = fig.add_subplot(gs[0, 2])
bars = ax3.bar(
    [f"k={int(r.k)}\npcs={int(r.pcs)}" for _, r in df_param_stability.iterrows()],
    df_param_stability['Spearman_r'],
    color=['#93C5FD', '#3B82F6', '#1D4ED8'],
    width=0.45,
    edgecolor='black',
    linewidth=0.8
)
ax3.set_ylim(0.8, 1.02)
ax3.set_ylabel("Spearman r vs Baseline", fontsize=10)
ax3.set_title("C. Graph Parameter Robustness", fontsize=11, fontweight='bold')
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
for bar in bars:
    yval = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2.0, yval + 0.005, f"{yval:.3f}", ha='center', va='bottom', fontsize=9, fontweight='bold')

# --- 子图 D: PAGA 拓扑网络叠加在 UMAP 上 (居中展示，比例均衡) ---
# 让 D 图跨占下方第 1、2 列或者全宽但保持 aspect 比例
ax4 = fig.add_subplot(gs[1, :])

# 底层 UMAP
sc.pl.umap(
    adata, 
    color='Final_Cell_Type', 
    ax=ax4, 
    show=False, 
    alpha=0.35, 
    size=25, 
    legend_loc=None
)

# 顶层 PAGA 连接网络
sc.pl.paga(
    adata,
    pos=umap_pos,
    color='Final_Cell_Type',
    ax=ax4,
    show=False,
    threshold=0.1,         # 隐藏弱假连接
    edge_width_scale=2.2,  # 适度调粗连线
    node_size_scale=3.5,   # 节点气泡大小
    fontoutline=4,         # 文字白色描边防重叠
    fontsize=13,
    title="D. PAGA Lineage Connectivity & Branch Uncertainty Assessment",
)

# 保持真实长宽比，防止挤压变形
ax4.set_aspect('equal', 'datalim')
for spine in ax4.spines.values():
    spine.set_visible(False)
ax4.set_xlabel("UMAP1", fontsize=11)
ax4.set_ylabel("UMAP2", fontsize=11)

plt.tight_layout()
plt.savefig("Supp_Figure_Trajectory_Robustness_Fixed.pdf", bbox_inches='tight')
plt.savefig("Supp_Figure_Trajectory_Robustness_Fixed.png", bbox_inches='tight')
plt.show()

print("✅ 图表已成功修复并导出！")

In [ ]:
# 1. 检查到底有多少细胞的拟时间堆在 0 附近
pt = adata.obs['dpt_pseudotime'].dropna()
print(f"拟时间 < 0.05 的细胞比例: {(pt < 0.05).mean() * 100:.2f}%")

# 2. 检查主根细胞到底是哪种细胞类型
print("当前主根细胞所在的 Cell Type:", adata.obs.loc[adata.obs_names[adata.uns['iroot']], 'Final_Cell_Type'])

# 3. 检查各细胞亚型的平均拟时间
print("\n各亚型的平均拟时间：")
print(adata.obs.groupby('Final_Cell_Type')['dpt_pseudotime'].mean().sort_values())

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "4"

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from scipy.spatial.distance import cdist

plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

OUTPUT_DIR = "arterial_remodeling_trajectory"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. 提取子集并【完全锁定】原图的全局空间坐标 (保持左下角分支形态)
# ------------------------------------------------------------
print("📥 1. 提取子集并保持原始全局拓扑空间...")
adata_full = sc.read_h5ad('adata_ec_clean2.h5ad')

target_subtypes = ['Arterial EC', 'ECM-remodeling Arterial EC']
adata_sub = adata_full[adata_full.obs['Final_Cell_Type'].isin(target_subtypes)].copy()
adata_sub.obs['Final_Cell_Type'] = adata_sub.obs['Final_Cell_Type'].astype('str').astype('category')

# 关键：直接锁定原大图的 X_umap 坐标，绝不重算 UMAP，保证位置与全局图 100% 对应！
# 重新计算该分支内部的高保真 diffusion map，只用于推断极性
sc.pp.neighbors(adata_sub, n_neighbors=15, n_pcs=20)
sc.tl.diffmap(adata_sub, n_comps=10)

# ------------------------------------------------------------
# 2. 精确锁定起点（远离 ECM-remodeling 的基线 Arterial EC 末端）
# ------------------------------------------------------------
print("🌱 2. 锚定真实的生物学发育起点...")
# 在 UMAP 坐标中，根据原图拓扑：ECM-remodeling 在左下角，起点应当选在远离它的右上侧 Arterial EC
ecm_mask = adata_sub.obs['Final_Cell_Type'] == 'ECM-remodeling Arterial EC'
ecm_center = np.median(adata_sub.obsm['X_umap'][ecm_mask], axis=0)

art_mask = adata_sub.obs['Final_Cell_Type'] == 'Arterial EC'
art_indices = np.where(art_mask)[0]

# 计算每个 Arterial 细胞到 ECM 重塑中心的物理流形距离
dists_to_ecm = cdist(adata_sub.obsm['X_umap'][art_indices], ecm_center.reshape(1, -1)).ravel()

# 选择距离 ECM-remodeling 最远（即最基线、最未活化）的 Arterial EC 细胞群作为 Root
# 取最远的前 10% 候选池中的稳健中位数
furthest_cands = art_indices[np.argsort(dists_to_ecm)[-int(len(art_indices)*0.1):]]
cand_diff = adata_sub.obsm['X_diffmap'][furthest_cands, 1:4]
cand_medoid = np.median(cand_diff, axis=0, keepdims=True)
main_root_idx = furthest_cands[np.argmin(cdist(cand_diff, cand_medoid))]

# 备选根用于稳健性检验
alt_roots = [
    furthest_cands[0],
    furthest_cands[len(furthest_cands)//2],
    furthest_cands[-1]
]

# 计算 DPT 拟时间
adata_sub.uns['iroot'] = main_root_idx
sc.tl.dpt(adata_sub, n_dcs=8)
base_pt = adata_sub.obs['dpt_pseudotime'].to_numpy().copy()

# 确保方向：起点必须在 Arterial，终点必须在 ECM-remodeling
mean_art = base_pt[art_mask].mean()
mean_ecm = base_pt[ecm_mask].mean()
if mean_art > mean_ecm:
    base_pt = 1.0 - base_pt
    adata_sub.obs['dpt_pseudotime'] = base_pt

# ------------------------------------------------------------
# 3. 根敏感性扰动检验
# ------------------------------------------------------------
print("🧪 3. 运行根细胞敏感性检验...")
alt_corrs = []
alt_pts = []
for alt_idx in alt_roots:
    ad_temp = adata_sub.copy()
    ad_temp.uns['iroot'] = alt_idx
    sc.tl.dpt(ad_temp, n_dcs=8)
    pt_alt = ad_temp.obs['dpt_pseudotime'].to_numpy().copy()
    r, _ = spearmanr(base_pt, pt_alt, nan_policy='omit')
    if r < 0:
        pt_alt = 1.0 - pt_alt
        r, _ = spearmanr(base_pt, pt_alt, nan_policy='omit')
    alt_corrs.append(r)
    alt_pts.append(pt_alt)

# ------------------------------------------------------------
# 4. 图参数敏感性检验 (KNN k=10, 15, 25)
# ------------------------------------------------------------
print("📊 4. 运行近邻参数扰动测试...")
param_grid = [{'k': 10, 'pcs': 15}, {'k': 15, 'pcs': 20}, {'k': 25, 'pcs': 25}]
param_corrs = []
for p in param_grid:
    ad_p = adata_sub.copy()
    sc.pp.neighbors(ad_p, n_neighbors=p['k'], n_pcs=p['pcs'])
    sc.tl.diffmap(ad_p, n_comps=10)
    ad_p.uns['iroot'] = main_root_idx
    sc.tl.dpt(ad_p, n_dcs=8)
    pt_p = ad_p.obs['dpt_pseudotime'].to_numpy()
    r_p, _ = spearmanr(base_pt, pt_p, nan_policy='omit')
    if r_p < 0:
        r_p, _ = spearmanr(base_pt, 1.0 - pt_p, nan_policy='omit')
    param_corrs.append(r_p)

# ------------------------------------------------------------
# 5. 绘制针对审稿人的规范级大图
# ------------------------------------------------------------
print("🎨 5. 正在导出高分辨率图表...")
fig = plt.figure(figsize=(15, 10), dpi=300)
gs = plt.GridSpec(2, 3, figure=fig, height_ratios=[1, 1], hspace=0.35, wspace=0.3)

# --- 子图 A: 亚型细胞类型分布 (锁定原图坐标，左下角分支清晰可见) ---
ax1 = fig.add_subplot(gs[0, 0])
sc.pl.umap(
    adata_sub, 
    color='Final_Cell_Type', 
    ax=ax1, 
    show=False, 
    title="A. Arterial Remodeling Lineage\n(Original Manifold)",
    frameon=False,
    palette=['#1E40AF', '#EA580C'],
    legend_loc='lower left'
)

# --- 子图 B: 平滑沿分支过渡的拟时间 ---
ax2 = fig.add_subplot(gs[0, 1])
sc.pl.umap(
    adata_sub, 
    color='dpt_pseudotime', 
    cmap='viridis', 
    ax=ax2, 
    show=False, 
    title="B. Pseudotime Trajectory (DPT)\n(Arterial -> ECM-remodeling)",
    frameon=False
)

# --- 子图 C: 箱线图量化两群的时间跃迁 ---
ax3 = fig.add_subplot(gs[0, 2])
pt_df = pd.DataFrame({
    'Subtype': adata_sub.obs['Final_Cell_Type'],
    'Pseudotime': adata_sub.obs['dpt_pseudotime']
})
pt_df.boxplot(
    column='Pseudotime', 
    by='Subtype', 
    ax=ax3, 
    patch_artist=True,
    grid=False,
    widths=0.45
)
ax3.set_title("C. Pseudotime Progression", fontsize=11, fontweight='bold')
ax3.set_xlabel("")
ax3.set_ylabel("Pseudotime (DPT)", fontsize=10)
ax3.set_xticklabels(['Arterial EC', 'ECM-remodeling\nArterial EC'], fontsize=9.5)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
fig.suptitle('')

# --- 子图 D: 备选根敏感性散点图 ---
ax4 = fig.add_subplot(gs[1, 0])
ax4.scatter(base_pt, alt_pts[0], s=14, color='#2563EB', alpha=0.5, rasterized=True)
ax4.plot([0, 1], [0, 1], 'r--', lw=1.5, label='Identity line')
ax4.set_xlabel("Pseudotime (Main Root)", fontsize=10)
ax4.set_ylabel("Pseudotime (Alternative Root)", fontsize=10)
ax4.set_title(f"D. Root Sensitivity Test\n(Spearman r = {alt_corrs[0]:.3f})", fontsize=11, fontweight='bold')
ax4.legend(frameon=False, loc='upper left', fontsize=9)
ax4.spines['top'].set_visible(False)
ax4.spines['right'].set_visible(False)
ax4.grid(True, linestyle='--', alpha=0.25)

# --- 子图 E: 参数敏感性条形图 ---
ax5 = fig.add_subplot(gs[1, 1])
param_labels = [f"k={p['k']}\npcs={p['pcs']}" for p in param_grid]
bars = ax5.bar(
    param_labels,
    param_corrs,
    color=['#93C5FD', '#3B82F6', '#1D4ED8'],
    width=0.45,
    edgecolor='black',
    linewidth=0.8
)
ax5.set_ylim(0.7, 1.05)
ax5.set_ylabel("Spearman r vs Baseline", fontsize=10)
ax5.set_title("E. Graph Parameter Robustness", fontsize=11, fontweight='bold')
ax5.spines['top'].set_visible(False)
ax5.spines['right'].set_visible(False)
for bar in bars:
    yval = bar.get_height()
    ax5.text(bar.get_x() + bar.get_width()/2.0, yval + 0.008, f"{yval:.3f}", ha='center', va='bottom', fontsize=9, fontweight='bold')

# --- 子图 F: PAGA 连接支持度 ---
ax6 = fig.add_subplot(gs[1, 2])
sc.tl.paga(adata_sub, groups='Final_Cell_Type')
sc.pl.paga(
    adata_sub,
    color='Final_Cell_Type',
    ax=ax6,
    show=False,
    edge_width_scale=3.5,
    node_size_scale=4.5,
    fontsize=11,
    title="F. Sub-lineage PAGA Support"
)
for spine in ax6.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/Supp_Figure_Arterial_Trajectory_Robustness.png", bbox_inches='tight')
plt.savefig(f"{OUTPUT_DIR}/Supp_Figure_Arterial_Trajectory_Robustness.pdf", bbox_inches='tight')
plt.show()

# 保存对齐原图坐标的高质量子集
adata_sub.write_h5ad(f"{OUTPUT_DIR}/adata_arterial_trajectory.h5ad")
print("🎉 修正完成！位置与全局 PAGA 完全吻合，单向流动渐变清晰！")

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from matplotlib.lines import Line2D

# 字体与排版设置
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

OUTPUT_DIR = "arterial_remodeling_trajectory"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. 读取单细胞轨迹数据
# ------------------------------------------------------------
adata = sc.read_h5ad(f"{OUTPUT_DIR}/adata_arterial_trajectory.h5ad")

if hasattr(adata.X, "toarray"):
    expr_mat = adata.X.toarray()
else:
    expr_mat = np.array(adata.X)

pt = adata.obs['dpt_pseudotime'].to_numpy()
subtypes = adata.obs['Final_Cell_Type'].to_numpy()

# ------------------------------------------------------------
# 2. 全基因组轨迹感知动力学建模与高动态基因筛选
# ------------------------------------------------------------
print("🔍 正在筛选高落差、高表达丰度的早期下调与晚期上调动态基因...")

# 划分早期区间 (DPT < 0.25) 与 晚期区间 (DPT > 0.65)
early_mask = pt < 0.25
late_mask = pt > 0.65

mean_early = expr_mat[early_mask].mean(axis=0)
mean_late = expr_mat[late_mask].mean(axis=0)
pct_early = (expr_mat[early_mask] > 0).mean(axis=0)
pct_late = (expr_mat[late_mask] > 0).mean(axis=0)

# 过滤无意义/预测序列基因
valid_name_mask = np.array([
    not (g.startswith('LOC') or g.startswith('Gm') or 'Rik' in g or g.startswith('AL') or g.startswith('AC') or g.startswith('MT-'))
    for g in adata.var_names
])

corrs, pvals = [], []
for j in range(expr_mat.shape[1]):
    r, p = spearmanr(pt, expr_mat[:, j])
    corrs.append(r)
    pvals.append(p)

_, fdr, _, _ = multipletests(pvals, method='fdr_bh')

stats_df = pd.DataFrame({
    'Gene': adata.var_names,
    'Spearman_r': corrs,
    'P_value': pvals,
    'GAM_FDR': fdr,
    'Mean_Early': mean_early,
    'Mean_Late': mean_late,
    'Pct_Early': pct_early,
    'Pct_Late': pct_late,
    'Delta_Drop': mean_early - mean_late,
    'Delta_Rise': mean_late - mean_early
})
stats_df = stats_df[valid_name_mask]

# 严格挑选早期下调基准基因 (丰度足、落差大)
down_cands = stats_df[
    (stats_df['Spearman_r'] < -0.30) & 
    (stats_df['Mean_Early'] > 0.9) & 
    (stats_df['Pct_Early'] > 0.25) &
    (stats_df['Delta_Drop'] > 0.7)
].copy()

if len(down_cands) < 6:
    down_cands = stats_df[
        (stats_df['Spearman_r'] < -0.28) & 
        (stats_df['Mean_Early'] > 0.6) & 
        (stats_df['Delta_Drop'] > 0.5)
    ].copy()

down_cands['Rank_Score'] = -down_cands['Spearman_r'] * down_cands['Delta_Drop']
top_down_genes = down_cands.sort_values(by='Rank_Score', ascending=False).head(6)['Gene'].tolist()

# 严格挑选晚期重塑上调基因
up_cands = stats_df[
    (stats_df['Spearman_r'] > 0.40) & 
    (stats_df['Mean_Late'] > 1.2) & 
    (stats_df['Delta_Rise'] > 1.0)
].copy()
up_cands['Rank_Score'] = up_cands['Spearman_r'] * up_cands['Delta_Rise']
top_up_genes = up_cands.sort_values(by='Rank_Score', ascending=False).head(6)['Gene'].tolist()

# 确保核心驱动基因在列表中
for p_g in ['FN1', 'TAGLN', 'GATA6']:
    if p_g in stats_df['Gene'].values and p_g not in top_up_genes:
        top_up_genes[-1] = p_g

print(f"✅ 挑选出的早期高落差下调基因: {top_down_genes}")
print(f"✅ 挑选出的晚期高落差上调基因: {top_up_genes}")

# ------------------------------------------------------------
# 3. 广义加性非参数平滑核回归 (GAM Kernel Regression)
# ------------------------------------------------------------
def fit_kernel_smooth(x, y, bandwidth=0.18, n_grid=200):
    grid_x = np.linspace(0, 1.0, n_grid)
    dist_sq = (grid_x[:, None] - x[None, :]) ** 2
    weights = np.exp(-dist_sq / (2 * (bandwidth ** 2)))
    
    weights_sum = weights.sum(axis=1, keepdims=True)
    weights_sum[weights_sum == 0] = 1e-12
    norm_weights = weights / weights_sum
    
    smooth_y = np.dot(norm_weights, y)
    
    diff_sq = (y[None, :] - smooth_y[:, None]) ** 2
    var_y = np.sum(norm_weights * diff_sq, axis=1)
    n_eff = 1.0 / np.sum(norm_weights ** 2, axis=1)
    se_y = np.sqrt(var_y / np.maximum(n_eff, 1.0))
    
    smooth_y = np.clip(smooth_y, 0, None)
    low_y = np.clip(smooth_y - 1.96 * se_y, 0, None)
    up_y = smooth_y + 1.96 * se_y
    
    return grid_x, smooth_y, low_y, up_y

# ------------------------------------------------------------
# 4. 规范绘制 2 行 6 列顶级图表 (标题升级为 GAM 统计量)
# ------------------------------------------------------------
fig, axes = plt.subplots(2, 6, figsize=(22, 7.5), dpi=300, sharex=True)

palette = {'Arterial EC': '#3B82F6', 'ECM-remodeling Arterial EC': '#F97316'}
c_scatter = [palette[st] for st in subtypes]

np.random.seed(42)
jitter = np.random.normal(0, 0.012, size=len(pt))

smooth_curves_dict = {'Pseudotime_Grid': np.linspace(0, 1.0, 200)}

# 第一行：6 个早期下调基因
for j, gene in enumerate(top_down_genes):
    ax = axes[0, j]
    g_idx = np.where(adata.var_names == gene)[0][0]
    y_raw = expr_mat[:, g_idx]
    
    # 散点
    ax.scatter(np.clip(pt + jitter, 0, 1), y_raw, c=c_scatter, s=10, alpha=0.22, rasterized=True)
    
    # GAM 非参数平滑曲线与 95% 置信区间
    gx, gy, gly, guy = fit_kernel_smooth(pt, y_raw, bandwidth=0.18)
    ax.plot(gx, gy, color='#1E40AF', lw=3.0)
    ax.fill_between(gx, gly, guy, color='#93C5FD', alpha=0.55)
    
    row = stats_df[stats_df['Gene'] == gene].iloc[0]
    fdr_txt = "< 1e-4" if row['GAM_FDR'] < 1e-4 else f"= {row['GAM_FDR']:.2e}"
    # 标题明确标注 GAM 统计显著性
    ax.set_title(f"{gene}\n(Early/Arterial)\nGAM: FDR {fdr_txt}, r = {row['Spearman_r']:.2f}", fontsize=10.5, fontweight='bold')
    
    if j == 0:
        ax.set_ylabel("Normalized Expression", fontsize=10.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(bottom=-0.1)
    ax.grid(True, linestyle='--', alpha=0.25)
    
    smooth_curves_dict[f'{gene}_GAM_Fit'] = gy
    smooth_curves_dict[f'{gene}_Lower_95CI'] = gly
    smooth_curves_dict[f'{gene}_Upper_95CI'] = guy

# 第二行：6 个晚期重塑上调基因
for j, gene in enumerate(top_up_genes):
    ax = axes[1, j]
    g_idx = np.where(adata.var_names == gene)[0][0]
    y_raw = expr_mat[:, g_idx]
    
    # 散点
    ax.scatter(np.clip(pt + jitter, 0, 1), y_raw, c=c_scatter, s=10, alpha=0.22, rasterized=True)
    
    # GAM 非参数平滑曲线与 95% 置信区间
    gx, gy, gly, guy = fit_kernel_smooth(pt, y_raw, bandwidth=0.18)
    ax.plot(gx, gy, color='#DC2626', lw=3.0)
    ax.fill_between(gx, gly, guy, color='#FCA5A5', alpha=0.55)
    
    row = stats_df[stats_df['Gene'] == gene].iloc[0]
    fdr_txt = "< 1e-4" if row['GAM_FDR'] < 1e-4 else f"= {row['GAM_FDR']:.2e}"
    # 标题明确标注 GAM 统计显著性
    ax.set_title(f"{gene}\n(Late/Remodeling)\nGAM: FDR {fdr_txt}, r = {row['Spearman_r']:.2f}", fontsize=10.5, fontweight='bold')
    
    if j == 0:
        ax.set_ylabel("Normalized Expression", fontsize=10.5)
    ax.set_xlabel("Pseudotime (DPT)", fontsize=10.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(bottom=-0.1)
    ax.grid(True, linestyle='--', alpha=0.25)
    
    smooth_curves_dict[f'{gene}_GAM_Fit'] = gy
    smooth_curves_dict[f'{gene}_Lower_95CI'] = gly
    smooth_curves_dict[f'{gene}_Upper_95CI'] = guy

legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Arterial EC', markerfacecolor='#3B82F6', markersize=7),
    Line2D([0], [0], marker='o', color='w', label='ECM-remodeling Arterial EC', markerfacecolor='#F97316', markersize=7),
    Line2D([0], [0], color='#1E40AF', lw=3.0, label='GAM Smooth Fit (Early Arterial Baseline)'),
    Line2D([0], [0], color='#DC2626', lw=3.0, label='GAM Smooth Fit (Late Remodeling Activation)')
]
fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 1.02), ncol=4, frameon=False, fontsize=10.5)

plt.tight_layout()
plt.subplots_adjust(top=0.91)

# ------------------------------------------------------------
# 5. 保存发表级文件套件
# ------------------------------------------------------------
print("💾 正在保存所有结果文件...")

# 5.1 图像文件
fig_png = f"{OUTPUT_DIR}/Figure_Pseudotime_Dynamics_12Genes.png"
fig_pdf = f"{OUTPUT_DIR}/Figure_Pseudotime_Dynamics_12Genes.pdf"
plt.savefig(fig_png, bbox_inches='tight')
plt.savefig(fig_pdf, bbox_inches='tight')
plt.show()

# 5.2 统计分析大表 (CSV，可作为 Supplementary Table)
stats_csv = f"{OUTPUT_DIR}/Pseudotime_Dynamics_Statistics.csv"
stats_df.sort_values(by='Spearman_r').to_csv(stats_csv, index=False)

# 5.3 GAM 平滑拟合曲线坐标数值矩阵 (CSV)
curves_df = pd.DataFrame(smooth_curves_dict)
curves_csv = f"{OUTPUT_DIR}/Pseudotime_Smooth_Curves_Data.csv"
curves_df.to_csv(curves_csv, index=False)

# 5.4 更新单细胞 AnnData
adata.var['trajectory_dynamic'] = False
selected_12 = top_down_genes + top_up_genes
adata.var.loc[adata.var_names.isin(selected_12), 'trajectory_dynamic'] = True
adata.uns['trajectory_top_down_genes'] = top_down_genes
adata.uns['trajectory_top_up_genes'] = top_up_genes

h5ad_out = f"{OUTPUT_DIR}/adata_arterial_trajectory_with_dynamics.h5ad"
adata.write_h5ad(h5ad_out)

print(f"""
🎉 全部 GAM 拟时序分析已完美生成并保存！
1. 图像输出:
   - PNG: {fig_png}
   - PDF (矢量图): {fig_pdf}
2. 附表与数据:
   - 全基因动力学统计表: {stats_csv}
   - 拟合曲线采样矩阵: {curves_csv}
3. AnnData 数据存档:
   - 标注后的 h5ad: {h5ad_out}
""")

In [ ]:
import os
import re
import json
import requests
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = "arterial_remodeling_trajectory"
os.makedirs(f"{OUTPUT_DIR}/enrichment", exist_ok=True)

# ============================================================
# 1. 智能加载基因列表 (优先读刚保存的 CSV，防内核重启)
# ============================================================
stats_csv_path = f"{OUTPUT_DIR}/Pseudotime_Dynamics_Statistics.csv"

if 'stats_df' in locals() or 'stats_df' in globals():
    print("✅ 从当前内存读取 stats_df...")
    df_stat = stats_df
elif os.path.exists(stats_csv_path):
    print(f"✅ 从已保存的文件中恢复数据: {stats_csv_path}")
    df_stat = pd.read_csv(stats_csv_path)
else:
    df_stat = None

if df_stat is not None:
    # 从动力学表中提取上/下调基因
    up_genes = df_stat[
        (df_stat['GAM_FDR'] < 1e-4) & 
        (df_stat['Spearman_r'] > 0.25) & 
        (df_stat['Mean_Late'] > 0.5)
    ].sort_values(by='Spearman_r', ascending=False)['Gene'].dropna().unique().tolist()

    down_genes = df_stat[
        (df_stat['GAM_FDR'] < 1e-4) & 
        (df_stat['Spearman_r'] < -0.25) & 
        (df_stat['Mean_Early'] > 0.5)
    ].sort_values(by='Spearman_r', ascending=True)['Gene'].dropna().unique().tolist()
elif 'up_genes_df' in locals() and 'down_genes_df' in locals():
    print("✅ 从 up_genes_df / down_genes_df 变量中读取基因...")
    up_genes = up_genes_df['Gene'].dropna().unique().tolist()
    down_genes = down_genes_df['Gene'].dropna().unique().tolist()
else:
    raise FileNotFoundError("未找到 stats_df 或历史 CSV 文件，请确认上一步脚本是否执行成功并生成了 CSV！")

print(f"📊 提取成功:")
print(f"  - 晚期重塑上调基因 (Late Remodeling): {len(up_genes)} 个")
print(f"  - 早期动脉下调基因 (Early Arterial):   {len(down_genes)} 个")

# ============================================================
# 2. 原生 Enrichr 请求接口 (免装 gseapy)
# ============================================================
def fetch_enrichr_res2d(gene_list, description="gene_list", gene_set='GO_Biological_Process_2023'):
    """无需本地安装 gseapy，直接调用 Enrichr API 并输出同结构结果"""
    if len(gene_list) < 5:
        print(f"⚠️ 基因数量过少 ({len(gene_list)} 个)，跳过请求。")
        return pd.DataFrame()

    print(f"⏳ 正在连接 Enrichr 获取 [{description}] 通路数据...")
    add_url = 'https://maayanlab.cloud/Enrichr/addList'
    payload = {'list': (None, '\n'.join(gene_list)), 'description': (None, description)}
    
    try:
        res = requests.post(add_url, files=payload, timeout=30)
        user_list_id = res.json().get('userListId')

        enrich_url = 'https://maayanlab.cloud/Enrichr/enrich'
        params = {'userListId': user_list_id, 'backgroundType': gene_set}
        res_data = requests.get(enrich_url, params=params, timeout=30).json().get(gene_set, [])

        records = []
        for row in res_data:
            overlap_genes = row[5]
            count = len(overlap_genes)
            records.append({
                'Term': row[1],
                'Count': count,
                'Overlap': f"{count}/{len(gene_list)}",  # 构建分数字符串以兼容后续计算
                'P-value': float(row[2]),
                'Adjusted P-value': float(row[6]),
                'Genes': ";".join(overlap_genes)
            })
            
        return pd.DataFrame(records)
    except Exception as e:
        print(f"❌ 请求失败: {e}")
        return pd.DataFrame()

# 获取富集数据表
up_enrich_res = fetch_enrichr_res2d(up_genes, description="EC_Late_Remodeling")
down_enrich_res = fetch_enrichr_res2d(down_genes, description="EC_Early_Arterial")

# ============================================================
# 3. 经典气泡图绘制函数 (完全遵循你的原版样式)
# ============================================================
def plot_enrichment_by_generatio(df, title, save_name, cmap='Reds', sort_by='Gene_Ratio'):
    """绘制富集分析气泡图"""
    if df is None or df.empty:
        print(f"没有可用于绘制 {title} 的富集数据。")
        return

    # 1. 过滤显著的通路
    plot_df = df[df['Adjusted P-value'] < 0.05].copy()
    p_col = 'Adjusted P-value'
    if plot_df.empty:
        plot_df = df[df['P-value'] < 0.05].copy()
        p_col = 'P-value'

    if plot_df.empty:
        print(f"⚠️ {title} 未筛选到 P < 0.05 的显著通路，将展示前10个最显著通路。")
        plot_df = df.copy()
        p_col = 'P-value'

    plot_df['-log10(P-val)'] = -np.log10(plot_df[p_col] + 1e-12)

    # 2. 提取 Gene Count 和 Gene Ratio
    plot_df['Count'] = plot_df['Overlap'].apply(lambda x: int(str(x).split('/')[0]))
    plot_df['Gene_Ratio'] = plot_df['Overlap'].apply(lambda x: eval(str(x)))

    # 3. 整理和清洗通路名称
    def clean_term(term):
        if ' (GO:' in term:
            term = term.split(' (GO:')[0]
        return "\n".join(textwrap.wrap(term, width=40))

    plot_df['Clean_Term'] = plot_df['Term'].apply(clean_term)

    # 4. 严格排序并取前 10 个通路
    plot_df = plot_df.sort_values(by='-log10(P-val)', ascending=False).head(10)

    if sort_by == 'Gene_Ratio':
        plot_df = plot_df.sort_values(by=['Gene_Ratio', '-log10(P-val)'], ascending=[True, True])
    else:
        plot_df = plot_df.sort_values(by='-log10(P-val)', ascending=True)

    plot_df['Clean_Term'] = pd.Categorical(plot_df['Clean_Term'], categories=plot_df['Clean_Term'], ordered=True)

    # 5. 开始绘图
    fig, ax = plt.subplots(figsize=(7.5, 6), dpi=300)

    sizes = plot_df['Count'] * 20 + 60

    sc = ax.scatter(
        x=plot_df['Gene_Ratio'],
        y=plot_df['Clean_Term'],
        s=sizes,
        c=plot_df['-log10(P-val)'],
        cmap=cmap,
        edgecolors='black',
        linewidths=0.8,
        zorder=3
    )

    ax.grid(axis='both', linestyle='--', alpha=0.4, zorder=0)
    ax.set_title(f"Enrichment: {title}", fontsize=13, weight='bold', pad=15)
    ax.set_xlabel("Gene Ratio", fontsize=11, weight='bold')
    ax.set_ylabel("", fontsize=11)
    ax.tick_params(axis='both', labelsize=10)

    # 图例 1. 颜色条
    cbar = fig.colorbar(sc, ax=ax, shrink=0.5, pad=0.08)
    cbar.set_label(f'-log10({p_col.replace(" ", "_")})', fontsize=9, weight='bold')
    cbar.ax.tick_params(labelsize=8)

    # 图例 2. 气泡大小图例
    counts = plot_df['Count'].unique()
    if len(counts) >= 4:
        legend_counts = np.linspace(counts.min(), counts.max(), 4, dtype=int)
    else:
        legend_counts = sorted(list(counts))

    legend_handles = []
    for count in legend_counts:
        legend_handles.append(
            plt.scatter([], [], s=count * 20 + 60, c='gray', alpha=0.6, edgecolors='black', linewidths=0.8)
        )

    ax.legend(
        handles=legend_handles,
        labels=[str(c) for c in legend_counts],
        title="Gene Count",
        loc="center left",
        bbox_to_anchor=(1.28, 0.25),
        frameon=False,
        title_fontsize=9,
        fontsize=9,
        labelspacing=1.8,
        handletextpad=1.2
    )

    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(save_name, bbox_inches='tight')
    plt.show()

# ============================================================
# 4. 执行出图
# ============================================================
plot_enrichment_by_generatio(
    up_enrich_res,
    title="Up-regulated Genes (Pseudotime)",
    save_name=f"{OUTPUT_DIR}/enrichment/Enrichment_EC_Up_Genes_Sorted.pdf",
    cmap='Reds',
    sort_by='Gene_Ratio'
)

plot_enrichment_by_generatio(
    down_enrich_res,
    title="Down-regulated Genes (Pseudotime)",
    save_name=f"{OUTPUT_DIR}/enrichment/Enrichment_EC_Down_Genes_Sorted.pdf",
    cmap='Blues',
    sort_by='Gene_Ratio'
)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# ================= 1. 挑选代表性基因 =================

rep_up_genes = ['FN1', 'GATA6', 'MFAP5', 'CHP2', 'LTBP4', 'TAGLN']
rep_down_genes = ['ADGRF5', 'MPPED2', 'EBF1', 'PBX1', 'CDC14A', 'ADGRL4']
# ================= 2. 创建 2行7列 的 UMAP 画布 =================
fig, axes = plt.subplots(2, 7, figsize=(28, 8), dpi=200)

umap_kwargs = {
    'show': False,
    'size': 15,            
    'frameon': False,      
    'use_raw': True        
}

# --- 1. 第一列: 背景对照 (Cell Type 和 Pseudotime) ---
cell_type_col = 'Final_Cell_Type' if 'Final_Cell_Type' in adata.obs.columns else 'cell_type'
if cell_type_col in adata.obs.columns:
    sc.pl.umap(
        adata, 
        color=cell_type_col, 
        ax=axes[0, 0], 
        title="Cell Type",
        legend_loc='on data',     # 🌟【核心修复1】将图例直接标在对应的细胞群上，彻底杜绝向右侧溢出重叠！
        legend_fontsize=10,       # 调整图例字体大小
        legend_fontoutline=2,     # 给图例加白色描边，防止看不清
        **umap_kwargs
    )
else:
    axes[0, 0].text(0.5, 0.5, "Cell Type Column\nNot Found", ha='center', va='center')
    axes[0, 0].axis('off')

# 左下: 拟时序数值 (Pseudotime)
sc.pl.umap(
    adata, 
    color='dpt_pseudotime', 
    cmap='viridis',        
    ax=axes[1, 0], 
    title="Pseudotime Trajectory",
    **umap_kwargs
)

# --- 2. 第一行 (后6列): 6个代表性上调基因 (Up-regulated) ---
for i, gene in enumerate(rep_up_genes):
    ax = axes[0, i + 1]
    sc.pl.umap(
        adata, 
        color=gene, 
        cmap='Reds',        
        ax=ax, 
        title=f"{gene}",
        **umap_kwargs
    )

# --- 3. 第二行 (后6列): 6个代表性下调基因 (Down-regulated) ---
for i, gene in enumerate(rep_down_genes):
    ax = axes[1, i + 1]
    sc.pl.umap(
        adata, 
        color=gene, 
        cmap='Blues',       
        ax=ax, 
        title=f"{gene}",
        **umap_kwargs
    )

# ================= 3. 细节美化与保存 =================
plt.suptitle("Expression of Pseudotime-associated Representative Genes on UMAP", 
             fontsize=18, weight='bold', y=1.02)

# 🌟【核心修复2】把列间距(wspace)加大到 0.6，给每个基因的 colorbar 留出充足呼吸空间
plt.subplots_adjust(wspace=0.6, hspace=0.3) 

plt.savefig('Pseudotime_Representative_Genes_UMAP_2x7_Final.pdf', bbox_inches='tight')
plt.show()
